# Detecção e Classificação de Malware em Redes IoT com WiSARD
## Aplicação ao Dataset TON_IoT Telemetry (Alsaedi et al., 2020)
### Versão com Pesos por Feature

### Contexto acadêmico

Este notebook implementa um sistema de **Intrusion Detection System (IDS)** baseado em **Redes Neurais Sem Peso (WNN)** para detecção e classificação de ataques em redes IoT. Utiliza o dataset **TON_IoT Telemetry** (Alsaedi et al., IEEE Access, 2020), que contém dados de telemetria de 7 sensores IoT com 9 tipos de ataques.

### Técnica de pesos por feature

Esta versão incorpora a técnica de **pesos hierárquicos por feature**.
Aplicada aqui ao TON_IoT: cada feature de cada dispositivo recebe um peso 1–5, e o `DynamicThermometer` gera `peso × BASE_THERMO_BITS` bits por feature. Os pesos vencedores foram determinados empiricamente em experimento prévio (notebook `wisard_toniot_feature_weights`), comparando 4 esquemas (`uniform`, `expert_v1`, `expert_v2`, `extreme`) por dispositivo e selecionando o melhor pela média de F1 nas 4 combinações (WiSARD/ClusWiSARD × binary/multiclass).

### Dois problemas tratados
- **Classificação binária**: normal vs ataqueFor large-scale testing, IoTFuzz employs digital twins to simulate normal behaviors, outdoor environment impacts, and human activities in SHs. Moreover, IoTFuzz can also intelligently infer rule-policy correlation based on natural language processing (NLP) techniques
- **Classificação multi-classe**: tipo específico de ataque (9 classes)


## 1. Instalação das Dependências

In [1]:
# wisardpkg — branch develop (API real com DynamicThermometer, ClusWisard, etc.)
!pip install -q --upgrade setuptools pybind11
!pip install -q git+https://github.com/IAZero/wisardpkg.git@develop
!pip install -q pandas scikit-learn matplotlib seaborn

import wisardpkg as wp
print('wisardpkg OK')
print('Modelos disponíveis:', [x for x in dir(wp) if not x.startswith('_')])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Preparing metadata (setup.py) ... done
wisardpkg OK
Modelos disponíveis: ['BestBleaching', 'BinBase', 'BinInput', 'Bleaching', 'ClassificationBase', 'ClassificationModel', 'ClusRegressionWisard', 'ClusWisard', 'DataSet', 'Discriminator', 'DynamicThermometer', 'ExponentialMean', 'GeometricMean', 'HarmonicMean', 'HarmonicPowerMean', 'KernelCanvas', 'LogisticMean', 'MappingGenerator', 'Mean', 'MeanThresholding', 'Median', 'Model', 'PowerMean', 'RAMDataHandle', 'RandomMapping', 'RegressionModel', 'RegressionRAMDataHandle', 'RegressionWisard', 'SimpleMean', 'SimpleThermometer', 'Synthesizer', 'Thresholdi

## 2. Carregamento dos Datasets



In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# CONFIGURAÇÃO — altere apenas este caminho
# Deve apontar para a pasta Train_Test_datasets no seu Drive
# Exemplo: '/content/drive/MyDrive/TON_IoT/Train_Test_datasets'
# ============================================================
DATASET_BASE_PATH = '/content/drive/MyDrive/TON_IoT/Train_Test_datasets'
# ============================================================

# Mapeamento dos arquivos por dispositivo
DEVICE_FILES = {
    'Fridge':       'Train_Test_IoT_Fridge.csv',
    'GPS_Tracker':  'Train_Test_IoT_GPS_Tracker.csv',
    'Garage_Door':  'Train_Test_IoT_Garage_Door.csv',
    'Motion_Light': 'Train_Test_IoT_Motion_Light.csv',
    'Modbus':       'Train_Test_IoT_Modbus.csv',
    'Thermostat':   'Train_Test_IoT_Thermostat.csv',
    'Weather':      'Train_Test_IoT_Weather.csv',
}

# Verificar quais arquivos estão disponíveis
print('Verificando arquivos no Drive:')
for name, fname in DEVICE_FILES.items():
    path = os.path.join(DATASET_BASE_PATH, fname)
    status = '✓' if os.path.exists(path) else '✗ NÃO ENCONTRADO'
    print(f'  {status}  {name}: {fname}')

Mounted at /content/drive
Verificando arquivos no Drive:
  ✓  Fridge: Train_Test_IoT_Fridge.csv
  ✓  GPS_Tracker: Train_Test_IoT_GPS_Tracker.csv
  ✓  Garage_Door: Train_Test_IoT_Garage_Door.csv
  ✓  Motion_Light: Train_Test_IoT_Motion_Light.csv
  ✓  Modbus: Train_Test_IoT_Modbus.csv
  ✓  Thermostat: Train_Test_IoT_Thermostat.csv
  ✓  Weather: Train_Test_IoT_Weather.csv


## 3. Imports e Configuração Global

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wisardpkg as wp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# HIPERPARÂMETROS GLOBAIS
# ============================================================
ADDRESS_SIZE     = 32     # bits por RAM (ajustado dinamicamente se não dividir N_BITS)
BASE_THERMO_BITS = 16     # bits para feature de peso 1 (peso k → k * BASE_THERMO_BITS bits)
THERMO_BITS      = BASE_THERMO_BITS  # alias mantido p/ compatibilidade com células abaixo
TEST_SIZE        = 0.20  # 80% treino / 20% teste (protocolo do artigo)
RANDOM_STATE     = 42

# ClusWiSARD
CLUS_MIN_SCORE   = 0.05   # score mínimo para pertencer a cluster existente (default=0.1)
CLUS_THRESHOLD   = 100   # exemplos máximos por discriminador
CLUS_LIMIT       = 20     # máximo de discriminadores por classe (default=5)

# Dispositivo padrão para demonstração individual
DEFAULT_DEVICE   = 'Weather'
# ============================================================

print('Configuração:')
print(f'  ADDRESS_SIZE={ADDRESS_SIZE}, BASE_THERMO_BITS={BASE_THERMO_BITS}')
print(f'  TEST_SIZE={TEST_SIZE}, RANDOM_STATE={RANDOM_STATE}')


def pick_address_size(n_bits, preferred=ADDRESS_SIZE):
    """Maior address_size <= preferred que divide n_bits.
    Como os pesos por feature mudam o n_bits, precisamos escolher
    um address_size compatível dinamicamente."""
    for a in range(preferred, 1, -1):
        if n_bits % a == 0:
            return a
    return 2


KeyboardInterrupt: 

## 4. Definição das Features por Dispositivo

Cada dispositivo IoT do TON_IoT possui features específicas (Alsaedi et al., 2020, Tables 2-8).
Removemos `date`, `time` e `ts` pois são metadados de registro, não features de comportamento.
Features categóricas são codificadas numericamente antes da binarização.

In [ ]:
# Features numéricas/binárias por dispositivo (excluindo date, time, ts, label, type)
DEVICE_FEATURES = {
    'Fridge':       ['fridge_temperature', 'temp_condition'],
    'GPS_Tracker':  ['latitude', 'longitude'],
    'Garage_Door':  ['door_state', 'sphone_signal'],
    'Motion_Light': ['motion_status', 'light_status'],
    'Modbus':       ['FC1_Read_Input_Register', 'FC2_Read_Discrete_Value',
                     'FC3_Read_Holding_Register', 'FC4_Read_Coil'],
    'Thermostat':   ['current_temperature', 'thermostat_status'],
    'Weather':      ['temperature', 'pressure', 'humidity'],
}

# Features categóricas que precisam de LabelEncoder antes da binarização
CATEGORICAL_FEATURES = {
    'Fridge':       ['temp_condition'],   # 'high' / 'low'
    'Garage_Door':  ['door_state', 'sphone_signal'],  # 'open'/'closed', True/False
    'Motion_Light': ['light_status'],     # 'on' / 'off'
    'Thermostat':   ['thermostat_status'],# True / False
    'GPS_Tracker':  [],
    'Modbus':       [],
    'Weather':      [],
}

# ============================================================
# PESOS POR FEATURE — VENCEDORES DO EXPERIMENTO DE PESOS
# ============================================================
# Selecionados em experimento prévio (notebook wisard_toniot_feature_weights),
# comparando 4 esquemas por dispositivo (uniform, expert_v1, expert_v2, extreme)
# e escolhendo o que maximiza a média de F1 entre as 4 combinações
# (WiSARD/ClusWiSARD × binary/multiclass).
#
# Esquema vencedor por dispositivo:
#   Fridge        -> expert_v1  (avg F1 = 0.7432, vs uniform 0.7402)
#   GPS_Tracker   -> expert_v2  (avg F1 = 0.6780, vs uniform 0.5682) -- maior ganho relativo
#   Garage_Door   -> uniform    (todos esquemas empatam; mantido baseline)
#   Motion_Light  -> uniform    (todos esquemas empatam; mantido baseline)
#   Modbus        -> expert_v2  (avg F1 = 0.4558, vs uniform 0.4363)
#   Thermostat    -> extreme    (avg F1 = 0.4695, vs uniform 0.4178)
#   Weather       -> expert_v2  (avg F1 = 0.7097, vs uniform 0.6591)
# ============================================================
BEST_WEIGHTS = {
    'Fridge':       {'fridge_temperature': 4, 'temp_condition': 2},                   # expert_v1
    'GPS_Tracker':  {'latitude': 5, 'longitude': 2},                                  # expert_v2
    'Garage_Door':  {'door_state': 3, 'sphone_signal': 3},                            # uniform
    'Motion_Light': {'motion_status': 3, 'light_status': 3},                          # uniform
    'Modbus':       {'FC1_Read_Input_Register': 5, 'FC2_Read_Discrete_Value': 3,
                     'FC3_Read_Holding_Register': 5, 'FC4_Read_Coil': 3},             # expert_v2
    'Thermostat':   {'current_temperature': 5, 'thermostat_status': 1},               # extreme
    'Weather':      {'temperature': 5, 'pressure': 4, 'humidity': 3},                 # expert_v2
}

print('Features e pesos por dispositivo:')
for dev, feats in DEVICE_FEATURES.items():
    w = BEST_WEIGHTS[dev]
    total = sum(w.values()) * BASE_THERMO_BITS
    print(f'  {dev:14s}: {dict((f, w[f]) for f in feats)} -> {total} bits/amostra')


## 5. Pré-Processamento — Fundamentos

O pipeline de pré-processamento segue o protocolo de Alsaedi et al. (2020):
1. **Encode categórico** — LabelEncoder para features nominais
2. **Normalização min-max** — escalar para [0, 1] (Eq. 15 do artigo)
3. **Binarização ponderada** — `DynamicThermometer` com `thermometerSizes` proporcionais
   aos pesos de `BEST_WEIGHTS`: peso $k$ ⇒ $k \times \texttt{BASE\_THERMO\_BITS}$ bits.
   Features mais discriminantes ocupam proporcionalmente mais bits no endereçamento das RAMs.

### Por que DynamicThermometer?
`DynamicThermometer` permite **tamanhos diferentes por feature** — ideal para
implementar a técnica de pesos. No notebook original todas as features recebiam
`THERMO_BITS=8` bits; aqui cada feature recebe `peso × 8` bits.


In [ ]:
def load_and_preprocess(device_name, task='binary', weights=None):
    """
    Carrega e pré-processa um dataset TON_IoT para um dispositivo, aplicando
    pesos por feature na binarização via DynamicThermometer.

    Args:
        device_name: nome do dispositivo (chave em DEVICE_FILES)
        task: 'binary' (normal vs ataque) ou 'multiclass' (tipo de ataque)
        weights: dict {feature_name: weight (int)}. Se None, usa BEST_WEIGHTS[device_name].

    Returns:
        X_train, X_test: listas de listas de bits (formato wisardpkg)
        y_train, y_test: listas de strings com os rótulos
        feature_names: lista de nomes das features
        label_info: dict com informações sobre as classes
    """
    # ── 1. Carregar CSV ──────────────────────────────────────────
    path = os.path.join(DATASET_BASE_PATH, DEVICE_FILES[device_name])
    df = pd.read_csv(path)
    print(f'[{device_name}] {len(df)} linhas carregadas')
    print(f'  Colunas: {list(df.columns)}')
    print(f'  Distribuição de classes (type):')
    print(df['type'].value_counts().to_string()); print()

    # ── 2. Selecionar features ───────────────────────────────────
    feature_cols = DEVICE_FEATURES[device_name]
    cat_cols     = CATEGORICAL_FEATURES.get(device_name, [])

    df_feat = df[feature_cols].copy()

    # ── 3. Encode categórico ─────────────────────────────────────
    le_dict = {}
    for col in cat_cols:
        if col in df_feat.columns:
            le = LabelEncoder()
            df_feat[col] = le.fit_transform(df_feat[col].astype(str))
            le_dict[col] = le

    # ── 4. Preencher NaN com mediana (protocolo do artigo) ───────
    df_feat = df_feat.fillna(df_feat.median(numeric_only=True))

    # ── 5. Normalização min-max → [0, 1] ─────────────────────────
    scaler = MinMaxScaler()
    X_norm = scaler.fit_transform(df_feat.values.astype(float))

    # ── 6. Definir rótulos ────────────────────────────────────────
    if task == 'binary':
        # 0 → 'normal', 1 → 'attack'
        y = df['label'].apply(lambda v: 'normal' if int(v) == 0 else 'attack').tolist()
    else:
        # tipo de ataque como string
        y = df['type'].astype(str).tolist()

    classes = sorted(set(y))
    label_info = {'classes': classes, 'n_classes': len(classes)}
    print(f'  Tarefa: {task} | Classes: {classes}')

    # ── 7. Train/Test split (80/20, estratificado) ────────────────
    X_tr_norm, X_te_norm, y_train, y_test = train_test_split(
        X_norm, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    # ── 8. Binarização PONDERADA com DynamicThermometer ──────────
    # Um termômetro por feature com tamanho proporcional ao peso da feature.
    # Esta é a aplicação direta da técnica de Galdino (2024) ao TON_IoT.
    if weights is None:
        weights = BEST_WEIGHTS[device_name]
    n_features = X_norm.shape[1]
    thermo_sizes = [weights[f] * BASE_THERMO_BITS for f in feature_cols]
    minimums    = [0.0] * n_features
    maximums    = [1.0] * n_features

    dy = wp.DynamicThermometer(
        thermometerSizes=thermo_sizes,
        minimum=minimums,
        maximum=maximums
    )

    print(f'  Pesos: {dict(zip(feature_cols, [weights[f] for f in feature_cols]))}')
    print(f'  Bits por feature: {dict(zip(feature_cols, thermo_sizes))}')
    print(f'  Binarizando {len(X_tr_norm)} amostras de treino...')
    X_train = [dy.transform(row.tolist()).list() for row in X_tr_norm]
    print(f'  Binarizando {len(X_te_norm)} amostras de teste...')
    X_test  = [dy.transform(row.tolist()).list() for row in X_te_norm]

    n_bits = len(X_train[0])
    print(f'  Bits por amostra: {n_bits} (total ponderado)')

    if n_bits % ADDRESS_SIZE != 0:
        print(f'  ℹ ADDRESS_SIZE={ADDRESS_SIZE} não divide n_bits={n_bits}; usaremos pick_address_size() onde for relevante.')

    return X_train, X_test, y_train, y_test, feature_cols, label_info


print('Função de pré-processamento (com pesos por feature) definida.')


## 6. Treinamento WiSARD — Classificação Binária

Testamos primeiro no dispositivo Weather, que possui 3 features contínuas
(temperatura, pressão, umidade) — um caso representativo dos dados de telemetria.

In [ ]:
# ── Carregar dispositivo padrão, tarefa binária ──────────────
print(f'=== Dispositivo: {DEFAULT_DEVICE} | Tarefa: Binária ===')
X_train, X_test, y_train, y_test, feat_names, label_info = load_and_preprocess(
    DEFAULT_DEVICE, task='binary'
)

# ── Instanciar e treinar WiSARD ──────────────────────────────
n_bits = len(X_train[0])
addr = pick_address_size(n_bits, preferred=ADDRESS_SIZE)
print(f'\nInstanciando WiSARD: addressSize={addr}, n_RAMs={n_bits//addr}')

model_bin = wp.Wisard(addr, bleachingActivated=True)

ds_train = wp.DataSet(X_train, y_train)
t0 = time.time()
model_bin.train(ds_train)
train_time = time.time() - t0
print(f'Treino concluído em {train_time:.3f}s')

# ── Classificar ──────────────────────────────────────────────
t0 = time.time()
y_pred = list(model_bin.classify(wp.DataSet(X_test)))
test_time = time.time() - t0
print(f'Teste concluído em {test_time:.3f}s')


## 7. Avaliação — Classificação Binária

In [ ]:
def evaluate(y_true, y_pred, model_name, task, train_time, test_time):
    """Calcula e exibe métricas seguindo o protocolo de Alsaedi et al. (2020)."""
    avg = 'binary' if task == 'binary' else 'weighted'
    pos = 'attack' if task == 'binary' else None

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=avg, pos_label=pos, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=avg, pos_label=pos, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=avg, pos_label=pos, zero_division=0)

    print(f'\n=== {model_name} [{task}] ===')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'  Train time: {train_time:.4f}s')
    print(f'  Test time : {test_time:.4f}s')
    print()
    print(classification_report(y_true, y_pred, zero_division=0))

    return {'model': model_name, 'task': task,
            'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1,
            'train_time': train_time, 'test_time': test_time}


result_bin = evaluate(y_test, y_pred, 'WiSARD', 'binary', train_time, test_time)

# ── Matriz de confusão ──────────────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=label_info['classes'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_info['classes'],
            yticklabels=label_info['classes'])
plt.title(f'Matriz de Confusão — WiSARD Binário ({DEFAULT_DEVICE})')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.tight_layout(); plt.savefig('confusion_binary.png', dpi=150)
plt.show()

## 8. Classificação Multi-Classe (tipo de ataque)

WiSARD é naturalmente multi-classe: cria um discriminador por classe
e a classificação é feita pela comparação dos scores de todos os discriminadores.

In [ ]:
print(f'=== Dispositivo: {DEFAULT_DEVICE} | Tarefa: Multi-classe ===')
X_train_mc, X_test_mc, y_train_mc, y_test_mc, _, label_info_mc = load_and_preprocess(
    DEFAULT_DEVICE, task='multiclass'
)

n_bits_mc = len(X_train_mc[0])
addr_mc = pick_address_size(n_bits_mc, preferred=ADDRESS_SIZE)
print(f'addressSize={addr_mc}, n_RAMs={n_bits_mc//addr_mc}')

model_mc = wp.Wisard(addr_mc, bleachingActivated=True)
ds_mc = wp.DataSet(X_train_mc, y_train_mc)

t0 = time.time()
model_mc.train(ds_mc)
train_time_mc = time.time() - t0

t0 = time.time()
y_pred_mc = list(model_mc.classify(wp.DataSet(X_test_mc)))
test_time_mc = time.time() - t0

result_mc = evaluate(y_test_mc, y_pred_mc, 'WiSARD', 'multiclass', train_time_mc, test_time_mc)

# ── Matriz de confusão ──────────────────────────────────────
cm_mc = confusion_matrix(y_test_mc, y_pred_mc, labels=label_info_mc['classes'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_info_mc['classes'],
            yticklabels=label_info_mc['classes'])
plt.title(f'Matriz de Confusão — WiSARD Multi-classe ({DEFAULT_DEVICE})')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig('confusion_multiclass.png', dpi=150)
plt.show()


## 9. Variante: ClusWiSARD

**ClusWiSARD** mantém múltiplos discriminadores por classe, capturando
subgrupos dentro de cada tipo de ataque — útil porque ataques como DDoS
podem ter padrões muito distintos em diferentes dispositivos IoT.

**API:**
```python
model = wp.ClusWisard(addressSize, minScore, threshold, discriminatorLimit)
```

In [ ]:
print(f'=== ClusWiSARD | {DEFAULT_DEVICE} | Multi-classe ===')

n_bits_mc = len(X_train_mc[0])
addr_clus = pick_address_size(n_bits_mc, preferred=ADDRESS_SIZE)
print(f'addressSize={addr_clus}')

model_clus = wp.ClusWisard(
    addr_clus,         # addressSize (dinâmico — depende dos pesos)
    CLUS_MIN_SCORE,    # minScore
    CLUS_THRESHOLD,    # threshold
    CLUS_LIMIT         # discriminatorLimit
)

t0 = time.time()
model_clus.train(wp.DataSet(X_train_mc, y_train_mc))
train_time_clus = time.time() - t0

t0 = time.time()
y_pred_clus = list(model_clus.classify(wp.DataSet(X_test_mc)))
test_time_clus = time.time() - t0

result_clus = evaluate(y_test_mc, y_pred_clus, 'ClusWiSARD', 'multiclass',
                       train_time_clus, test_time_clus)


## 10. Avaliação em Todos os Dispositivos

Replicamos o protocolo de Alsaedi et al. (2020) para todos os 7 dispositivos,
comparando WiSARD e ClusWiSARD em classificação binária e multi-classe.

In [ ]:
all_results = []

for device in DEVICE_FILES.keys():
    print(f'\n{"="*60}')
    print(f'Dispositivo: {device}')
    print('='*60)

    for task in ['binary', 'multiclass']:
        try:
            Xtr, Xte, ytr, yte, _, linfo = load_and_preprocess(device, task)
            n_bits = len(Xtr[0])
            addr = pick_address_size(n_bits, preferred=ADDRESS_SIZE)
            print(f'  [{task}] n_bits={n_bits}, addressSize={addr}')

            # ── WiSARD ──────────────────────────────────────────
            m = wp.Wisard(addr, bleachingActivated=True)
            t0 = time.time(); m.train(wp.DataSet(Xtr, ytr)); tt = time.time()-t0
            t0 = time.time(); yp = list(m.classify(wp.DataSet(Xte))); te = time.time()-t0
            r = evaluate(yte, yp, 'WiSARD', task, tt, te)
            r['device'] = device
            all_results.append(r)

            # ── ClusWiSARD ──────────────────────────────────────
            mc = wp.ClusWisard(addr, CLUS_MIN_SCORE, CLUS_THRESHOLD, CLUS_LIMIT)
            t0 = time.time(); mc.train(wp.DataSet(Xtr, ytr)); tt = time.time()-t0
            t0 = time.time(); yp = list(mc.classify(wp.DataSet(Xte))); te = time.time()-t0
            r = evaluate(yte, yp, 'ClusWiSARD', task, tt, te)
            r['device'] = device
            all_results.append(r)

        except FileNotFoundError:
            print(f'  [skip] arquivo não encontrado para {device}')
        except Exception as e:
            print(f'  [erro {task}] {e}')


## 11. Tabela de Resultados Consolidada

In [ ]:
import pandas as pd

df_results = pd.DataFrame(all_results)

# ── Tabela binária ──────────────────────────────────────────
print('=== Classificação Binária (normal vs ataque) ===')
df_bin = df_results[df_results['task']=='binary'][
    ['device','model','accuracy','precision','recall','f1','train_time','test_time']
].round(4)
print(df_bin.to_string(index=False))

print('\n=== Classificação Multi-classe (tipo de ataque) ===')
df_mc = df_results[df_results['task']=='multiclass'][
    ['device','model','accuracy','precision','recall','f1','train_time','test_time']
].round(4)
print(df_mc.to_string(index=False))

# Salvar CSV
df_results.to_csv('results_wisard_toniot.csv', index=False)
print('\nResultados salvos em results_wisard_toniot.csv')

## 12. Dataset Combinado (combined_IoT_dataset)

Seguindo Alsaedi et al. (2020), combinamos todos os dispositivos em um único dataset.
Features ausentes em cada dispositivo são preenchidas com a mediana.

In [ ]:
def load_combined(task='binary'):
    """
    Carrega e combina todos os dispositivos em um único dataset.
    Segue o protocolo de Alsaedi et al. (2020): mediana para NaN.

    Os pesos por feature são herdados de BEST_WEIGHTS (mesmos pesos vencedores).
    Features ausentes em um dispositivo recebem peso 1 por padrão.
    """
    all_features = set()
    for feats in DEVICE_FEATURES.values():
        all_features.update(feats)
    all_features = sorted(all_features)

    dfs = []
    for device in DEVICE_FILES.keys():
        path = os.path.join(DATASET_BASE_PATH, DEVICE_FILES[device])
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)

        # Encode categórico
        for col in CATEGORICAL_FEATURES.get(device, []):
            if col in df.columns:
                df[col] = LabelEncoder().fit_transform(df[col].astype(str))

        # Selecionar apenas features numéricas do dispositivo + labels
        feat_cols = [f for f in DEVICE_FEATURES[device] if f in df.columns]
        sub = df[feat_cols + ['label', 'type']].copy()
        sub['device'] = device
        dfs.append(sub)

    combined = pd.concat(dfs, ignore_index=True)

    # Preencher NaN com mediana por coluna (protocolo do artigo)
    numeric_cols = [c for c in combined.columns if c not in ['label','type','device']]
    for col in numeric_cols:
        combined[col] = pd.to_numeric(combined[col], errors='coerce')
        combined[col] = combined[col].fillna(combined[col].median())

    print(f'Combined dataset: {len(combined)} linhas, {len(numeric_cols)} features')
    print('Distribuição:', combined['type'].value_counts().to_dict())

    # Normalização
    X_raw = combined[numeric_cols].values.astype(float)
    X_norm = MinMaxScaler().fit_transform(X_raw)

    if task == 'binary':
        y = combined['label'].apply(lambda v: 'normal' if int(v)==0 else 'attack').tolist()
    else:
        y = combined['type'].astype(str).tolist()

    X_tr_n, X_te_n, y_tr, y_te = train_test_split(
        X_norm, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

    # ── Pesos para o combinado: lookup em BEST_WEIGHTS, default peso 1 ──
    combined_weights = []
    for f in numeric_cols:
        peso = 1
        for dev, w in BEST_WEIGHTS.items():
            if f in w:
                peso = max(peso, w[f])  # maior peso vencedor encontrado
                break
        combined_weights.append(peso)
    print(f'Pesos no combinado: {dict(zip(numeric_cols, combined_weights))}')

    thermo_sizes = [p * BASE_THERMO_BITS for p in combined_weights]
    n_feat = X_norm.shape[1]
    dy = wp.DynamicThermometer(
        thermometerSizes=thermo_sizes,
        minimum=[0.0]*n_feat,
        maximum=[1.0]*n_feat
    )

    print('Binarizando...')
    X_tr = [dy.transform(row.tolist()).list() for row in X_tr_n]
    X_te = [dy.transform(row.tolist()).list() for row in X_te_n]
    print(f'Bits por amostra: {len(X_tr[0])}')

    return X_tr, X_te, y_tr, y_te


# Avaliar dataset combinado
for task in ['binary', 'multiclass']:
    print(f'\n=== Combined IoT Dataset | {task} ===')
    Xtr_c, Xte_c, ytr_c, yte_c = load_combined(task)
    n_bits_c = len(Xtr_c[0])
    addr_c = pick_address_size(n_bits_c, preferred=ADDRESS_SIZE)
    print(f'addressSize={addr_c}')

    m_c = wp.Wisard(addr_c, bleachingActivated=True)
    t0 = time.time(); m_c.train(wp.DataSet(Xtr_c, ytr_c)); tt = time.time()-t0
    t0 = time.time(); yp_c = list(m_c.classify(wp.DataSet(Xte_c))); te = time.time()-t0
    evaluate(yte_c, yp_c, 'WiSARD', task, tt, te)


## 13. Análise de Sensibilidade — Address Size

In [ ]:
# Determinar ADDRESS_SIZEs compatíveis com os bits disponíveis
# (n_bits agora depende dos pesos por feature)
n_bits_sample = len(X_train[0])
valid_sizes = [a for a in [2, 4, 5, 8, 10, 14, 16, 20, 24, 28, 32, 40] if n_bits_sample % a == 0]
print(f'N_BITS={n_bits_sample} | ADDRESS_SIZEs compatíveis: {valid_sizes}')

results_sensitivity = []
for addr in valid_sizes:
    m = wp.Wisard(addr, bleachingActivated=True)
    m.train(wp.DataSet(X_train, y_train))
    yp = list(m.classify(wp.DataSet(X_test)))
    f1 = f1_score(y_test, yp, average='binary', pos_label='attack', zero_division=0)
    acc = accuracy_score(y_test, yp)
    results_sensitivity.append({'address_size': addr, 'f1': f1, 'accuracy': acc})
    print(f'  addressSize={addr:2d} -> acc={acc:.4f}, f1={f1:.4f}')

df_sens = pd.DataFrame(results_sensitivity)
plt.figure(figsize=(7,4))
plt.plot(df_sens['address_size'], df_sens['f1'], 'o-', label='F1', linewidth=2)
plt.plot(df_sens['address_size'], df_sens['accuracy'], 's--', label='Accuracy', linewidth=2)
plt.xlabel('Address Size')
plt.ylabel('Métrica')
plt.title(f'Sensibilidade ao Address Size ({DEFAULT_DEVICE}, binário, pesos vencedores)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('sensitivity_address.png', dpi=150)
plt.show()


## 14. Comparação com Baseline do Artigo

Alsaedi et al. (2020) avaliam 8 métodos: LR, LDA, kNN, RF, CART, NB, SVM, LSTM.
Os melhores resultados reportados para o dataset Weather (mais próximo ao nosso experimento):
- CART: Acc=0.87, Prec=0.88, Rec=0.87, F1=0.87
- RF:   Acc=0.84, Prec=0.84, Rec=0.84, F1=0.84

In [ ]:
# Tabela comparativa com baseline do artigo (Weather dataset, binário)
baseline_weather = pd.DataFrame([
    {'Modelo': 'LR',          'Accuracy': 0.58, 'Precision': 0.60, 'Recall': 0.59, 'F1': 0.53},
    {'Modelo': 'LDA',         'Accuracy': 0.60, 'Precision': 0.59, 'Recall': 0.60, 'F1': 0.53},
    {'Modelo': 'kNN',         'Accuracy': 0.81, 'Precision': 0.81, 'Recall': 0.81, 'F1': 0.81},
    {'Modelo': 'RF',          'Accuracy': 0.84, 'Precision': 0.84, 'Recall': 0.84, 'F1': 0.84},
    {'Modelo': 'CART',        'Accuracy': 0.87, 'Precision': 0.88, 'Recall': 0.87, 'F1': 0.87},
    {'Modelo': 'NB',          'Accuracy': 0.69, 'Precision': 0.72, 'Recall': 0.69, 'F1': 0.67},
    {'Modelo': 'SVM',         'Accuracy': 0.63, 'Precision': 0.68, 'Recall': 0.63, 'F1': 0.55},
    {'Modelo': 'LSTM',        'Accuracy': 0.82, 'Precision': 0.82, 'Recall': 0.81, 'F1': 0.80},
    {'Modelo': 'WiSARD',      'Accuracy': result_bin['accuracy'],
                               'Precision': result_bin['precision'],
                               'Recall':    result_bin['recall'],
                               'F1':        result_bin['f1']},
    {'Modelo': 'ClusWiSARD',  'Accuracy': result_clus['accuracy'],
                               'Precision': result_clus['precision'],
                               'Recall':    result_clus['recall'],
                               'F1':        result_clus['f1']},
]).round(4)

print(f'=== Comparação — {DEFAULT_DEVICE} | Tarefa binária ===')
print(baseline_weather.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#B5D4F4']*8 + ['#534AB7', '#1D9E75']
bars = ax.bar(baseline_weather['Modelo'], baseline_weather['F1'], color=colors)
ax.axhline(y=0.87, color='red', linestyle='--', alpha=0.5, label='CART (melhor baseline)')
ax.set_ylabel('F1-Score'); ax.set_ylim(0, 1.05)
ax.set_title(f'F1-Score — {DEFAULT_DEVICE} — Binário')
ax.legend(); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig('comparison_baseline.png', dpi=150)
plt.show()

## 15. Imagens Mentais — Interpretabilidade

Uma vantagem única do WiSARD: as **imagens mentais** (`getMentalImages()`)
revelam o que o modelo "aprendeu" como padrão típico de cada tipo de ataque.
Para dados tabulares, isso se traduz em um vetor de ativação por feature.

In [ ]:
def plot_mental_images_tabular(model, feature_names, weights, title='Imagens Mentais WiSARD'):
    """
    Visualiza as imagens mentais do WiSARD como heatmap de ativação por feature.
    Cada linha = uma classe, cada coluna = uma feature (média de ativação dos
    bits ponderados daquela feature).

    Args:
        weights: dict {feature_name: weight} usado na binarização
    """
    images = model.getMentalImages()
    classes = list(images.keys())
    n_feat  = len(feature_names)

    # Tamanho de cada feature em bits, na ordem de feature_names
    bits_per_feat = [weights[f] * BASE_THERMO_BITS for f in feature_names]

    # Colapsar bits por feature (média de ativação dentro do bloco de cada feature)
    heatmap = []
    for cls in classes:
        mi = np.array(images[cls])  # vetor 1D com a imagem mental
        row = []
        offset = 0
        for nbits in bits_per_feat:
            seg = mi[offset:offset+nbits]
            row.append(float(np.mean(seg)) if len(seg) > 0 else 0.0)
            offset += nbits
        heatmap.append(row)

    heatmap = np.array(heatmap)
    plt.figure(figsize=(max(6, n_feat*1.5), max(3, len(classes)*0.5)))
    sns.heatmap(heatmap, annot=True, fmt='.2f', cmap='viridis',
                xticklabels=feature_names, yticklabels=classes,
                cbar_kws={'label': 'Ativação média'})
    plt.title(title)
    plt.xlabel('Feature'); plt.ylabel('Classe')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('mental_images.png', dpi=150)
    plt.show()


# Plotar imagens mentais do modelo multi-classe
plot_mental_images_tabular(
    model_mc, feat_names,
    weights=BEST_WEIGHTS[DEFAULT_DEVICE],
    title=f'Imagens Mentais — WiSARD Multi-classe ({DEFAULT_DEVICE}, pesos vencedores)'
)


## 16. Comparação de Binarizações

Testamos três estratégias de binarização para avaliar qual preserva
melhor a informação discriminante dos ataques IoT:

| Método | Descrição |
|--------|-----------|
| **SimpleThermometer** | Um único range para todos os valores, sem pesos |
| **DynamicThermometer ponderado** | Bits por feature proporcionais a `BEST_WEIGHTS` (vencedor) |
| **Threshold global** | Binarização simples pela mediana |

A versão ponderada deve superar as outras nos dispositivos onde a técnica de pesos
demonstrou ganho no experimento prévio (GPS_Tracker, Thermostat, Weather, Modbus).


In [ ]:
# Recarregar dados normalizados (sem binarizar)
path = os.path.join(DATASET_BASE_PATH, DEVICE_FILES[DEFAULT_DEVICE])
df_raw = pd.read_csv(path)
feat_cols = DEVICE_FEATURES[DEFAULT_DEVICE]
X_raw_vals = df_raw[feat_cols].fillna(df_raw[feat_cols].median()).values.astype(float)
y_all = df_raw['label'].apply(lambda v: 'normal' if int(v)==0 else 'attack').tolist()
X_norm_all = MinMaxScaler().fit_transform(X_raw_vals)

X_tr_n, X_te_n, y_tr, y_te = train_test_split(
    X_norm_all, y_all, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_all
)

bin_results = []

# ── 1. SimpleThermometer (sem pesos, todas as features mesmo número de bits) ──
n_feat = X_norm_all.shape[1]
simple_bits = BASE_THERMO_BITS
st = wp.DynamicThermometer(
    thermometerSizes=[simple_bits]*n_feat,
    minimum=[0.0]*n_feat, maximum=[1.0]*n_feat
)
Xtr_s = [st.transform(r.tolist()).list() for r in X_tr_n]
Xte_s = [st.transform(r.tolist()).list() for r in X_te_n]
addr_s = pick_address_size(len(Xtr_s[0]), ADDRESS_SIZE)
m_s = wp.Wisard(addr_s, bleachingActivated=True)
m_s.train(wp.DataSet(Xtr_s, y_tr))
yp_s = list(m_s.classify(wp.DataSet(Xte_s)))
f1_s = f1_score(y_te, yp_s, average='binary', pos_label='attack', zero_division=0)
bin_results.append({'método': 'SimpleThermometer (sem pesos)', 'n_bits': len(Xtr_s[0]),
                    'addr': addr_s, 'F1': round(f1_s, 4)})

# ── 2. DynamicThermometer PONDERADO (pesos vencedores) ──
weights = BEST_WEIGHTS[DEFAULT_DEVICE]
thermo_sizes = [weights[f] * BASE_THERMO_BITS for f in feat_cols]
dt = wp.DynamicThermometer(
    thermometerSizes=thermo_sizes,
    minimum=[0.0]*n_feat, maximum=[1.0]*n_feat
)
Xtr_d = [dt.transform(r.tolist()).list() for r in X_tr_n]
Xte_d = [dt.transform(r.tolist()).list() for r in X_te_n]
addr_d = pick_address_size(len(Xtr_d[0]), ADDRESS_SIZE)
m_d = wp.Wisard(addr_d, bleachingActivated=True)
m_d.train(wp.DataSet(Xtr_d, y_tr))
yp_d = list(m_d.classify(wp.DataSet(Xte_d)))
f1_d = f1_score(y_te, yp_d, average='binary', pos_label='attack', zero_division=0)
bin_results.append({'método': f'DynamicThermometer ponderado {weights}',
                    'n_bits': len(Xtr_d[0]), 'addr': addr_d, 'F1': round(f1_d, 4)})

# ── 3. Threshold global (mediana) ──
X_thr_tr = (X_tr_n > 0.5).astype(int).tolist()
X_thr_te = (X_te_n > 0.5).astype(int).tolist()
# Cada amostra tem n_feat bits — precisamos expandir para ficar divisível
# Replicamos cada bit BASE_THERMO_BITS vezes para acomodar address_size
X_thr_tr_ext = [[b for b in row for _ in range(BASE_THERMO_BITS)] for row in X_thr_tr]
X_thr_te_ext = [[b for b in row for _ in range(BASE_THERMO_BITS)] for row in X_thr_te]
addr_t = pick_address_size(len(X_thr_tr_ext[0]), ADDRESS_SIZE)
m_t = wp.Wisard(addr_t, bleachingActivated=True)
m_t.train(wp.DataSet(X_thr_tr_ext, y_tr))
yp_t = list(m_t.classify(wp.DataSet(X_thr_te_ext)))
f1_t = f1_score(y_te, yp_t, average='binary', pos_label='attack', zero_division=0)
bin_results.append({'método': 'Threshold global (mediana)',
                    'n_bits': len(X_thr_tr_ext[0]), 'addr': addr_t, 'F1': round(f1_t, 4)})

df_bin_cmp = pd.DataFrame(bin_results)
print(f'=== Comparação de binarizações — {DEFAULT_DEVICE} (binário) ===')
print(df_bin_cmp.to_string(index=False))


## 17. Tabela Final de Resultados

In [ ]:
print('=== Sumário Final ===')
print(f'Dataset: TON_IoT Telemetry (Alsaedi et al., 2020)')
print(f'Modelos: WiSARD, ClusWiSARD')
print(f'Binarização: DynamicThermometer PONDERADO (técnica de Galdino, 2024)')
print(f'BASE_THERMO_BITS: {BASE_THERMO_BITS} (peso k -> k * {BASE_THERMO_BITS} bits)')
print(f'Address Size base: {ADDRESS_SIZE} (ajustado via pick_address_size)')
print(f'Split: {int((1-TEST_SIZE)*100)}% treino / {int(TEST_SIZE*100)}% teste')
print()

print('Pesos vencedores aplicados (selecionados em experimento prévio):')
for dev, w in BEST_WEIGHTS.items():
    print(f'  {dev:14s}: {w}')
print()

if all_results:
    df_final = pd.DataFrame(all_results)
    print('── Binário (média por modelo) ──')
    print(df_final[df_final['task']=='binary'].groupby('model')[['accuracy','precision','recall','f1']].mean().round(4))
    print()
    print('── Multi-classe (média por modelo) ──')
    print(df_final[df_final['task']=='multiclass'].groupby('model')[['accuracy','precision','recall','f1']].mean().round(4))
    print()
    print('── Top 3 dispositivos por F1 (binário, WiSARD) ──')
    top = df_final[(df_final['task']=='binary') & (df_final['model']=='WiSARD')].nlargest(3, 'f1')
    print(top[['device','accuracy','precision','recall','f1']].to_string(index=False))


# ============================================================
# OVERSAMPLING — Tratamento de desbalanceamento (multi-classe)
# Cole esta célula AO FINAL do notebook wisard_toniot_malware_detection.ipynb
# Requer que load_and_preprocess(), load_combined(), evaluate() e BEST_WEIGHTS
# já estejam definidos no notebook (são das células anteriores).
#
# Estratégia: oversampling por replicação simples no Y_train
#   - Antes da binarização termométrica
#   - Replica amostras das classes minoritárias até igualar a majoritária
#   - Apenas para tarefa multi-classe (binário não precisa)
# ============================================================

import numpy as np
import pandas as pd
import time
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import wisardpkg as wp


def oversample_replication(X, y, random_state=42):
    """
    Oversampling por replicação: duplica amostras das classes minoritárias
    até todas terem o mesmo tamanho da classe majoritária.

    X: np.ndarray (n, d) ou lista de listas - features normalizadas (pré-binarização)
    y: lista de strings com rótulos

    Retorna X_bal, y_bal já embaralhados.
    """
    X = np.asarray(X)
    y = np.asarray(y)
    counts = Counter(y)
    max_count = max(counts.values())
    print(f'   Distribuição original: {dict(counts)}')
    print(f'   Alvo por classe: {max_count}')

    rng = np.random.RandomState(random_state)
    X_parts, y_parts = [], []
    for cls, n in counts.items():
        idx = np.where(y == cls)[0]
        if n < max_count:
            # Replicar (com reposição) para atingir max_count
            extra_idx = rng.choice(idx, size=max_count - n, replace=True)
            full_idx = np.concatenate([idx, extra_idx])
        else:
            full_idx = idx
        X_parts.append(X[full_idx])
        y_parts.append(y[full_idx])

    X_bal = np.vstack(X_parts)
    y_bal = np.concatenate(y_parts)

    # Embaralhar
    perm = rng.permutation(len(y_bal))
    X_bal, y_bal = X_bal[perm], y_bal[perm]

    print(f'   Distribuição após oversampling: {dict(Counter(y_bal))}')
    print(f'   Tamanho total: {len(y_bal)} (antes: {len(y)})')
    return X_bal, y_bal.tolist()


def run_oversampling_experiment_device(device_name, models=('WiSARD', 'ClusWiSARD')):
    """
    Roda oversampling em um dispositivo (multi-classe apenas).
    Retorna lista de dicts com resultados.
    """
    print(f'\n{"="*70}')
    print(f'OVERSAMPLING EXPERIMENT: {device_name} | multi-classe')
    print(f'{"="*70}')

    # ── Carregar pré-binarizado (precisamos refazer para acessar X_train normalizado) ──
    path = os.path.join(DATASET_BASE_PATH, DEVICE_FILES[device_name])
    df = pd.read_csv(path)
    feature_cols = DEVICE_FEATURES[device_name]
    cat_cols = CATEGORICAL_FEATURES.get(device_name, [])

    df_feat = df[feature_cols].copy()
    for col in cat_cols:
        if col in df_feat.columns:
            df_feat[col] = LabelEncoder().fit_transform(df_feat[col].astype(str))
    df_feat = df_feat.fillna(df_feat.median(numeric_only=True))

    scaler = MinMaxScaler()
    X_norm = scaler.fit_transform(df_feat.values.astype(float))
    y = df['type'].astype(str).tolist()

    X_tr_n, X_te_n, y_tr, y_te = train_test_split(
        X_norm, y, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=y)

    # ── OVERSAMPLING APENAS NO TREINO ──
    print(f'\n[{device_name}] Aplicando oversampling no conjunto de treino...')
    X_tr_bal, y_tr_bal = oversample_replication(X_tr_n, y_tr, random_state=RANDOM_STATE)

    # ── Binarização (usando pesos vencedores) ──
    weights = BEST_WEIGHTS[device_name]
    thermo_sizes = [weights[f] * BASE_THERMO_BITS for f in feature_cols]
    n_feat = len(feature_cols)
    dy = wp.DynamicThermometer(
        thermometerSizes=thermo_sizes,
        minimum=[0.0]*n_feat, maximum=[1.0]*n_feat)

    print(f'   Binarizando {len(X_tr_bal)} amostras de treino balanceadas...')
    X_tr_bin = [dy.transform(row.tolist()).list() for row in X_tr_bal]
    X_te_bin = [dy.transform(row.tolist()).list() for row in X_te_n]
    n_bits = len(X_tr_bin[0])
    addr = pick_address_size(n_bits, preferred=ADDRESS_SIZE)
    print(f'   N_BITS={n_bits}, addressSize={addr}')

    results = []
    for model_name in models:
        print(f'\n--- Treinando {model_name} (oversampled) ---')
        if model_name == 'WiSARD':
            m = wp.Wisard(addr, bleachingActivated=True)
        else:  # ClusWiSARD
            m = wp.ClusWisard(addr, CLUS_MIN_SCORE, CLUS_THRESHOLD, CLUS_LIMIT)

        t0 = time.time()
        m.train(wp.DataSet(X_tr_bin, y_tr_bal))
        train_time = time.time() - t0

        t0 = time.time()
        y_pred = list(m.classify(wp.DataSet(X_te_bin)))
        test_time = time.time() - t0

        r = evaluate(y_te, y_pred, f'{model_name}-OS', 'multiclass', train_time, test_time)
        r['device'] = device_name
        r['oversampled'] = True
        results.append(r)

    return results


def run_oversampling_experiment_combined(models=('WiSARD', 'ClusWiSARD')):
    """
    Roda oversampling no dataset combinado (multi-classe apenas).
    """
    print(f'\n{"="*70}')
    print(f'OVERSAMPLING EXPERIMENT: Combined | multi-classe')
    print(f'{"="*70}')

    # Reconstruir combined pré-binarizado (igual load_combined mas sem binarizar)
    all_features = set()
    for feats in DEVICE_FEATURES.values():
        all_features.update(feats)

    dfs = []
    for device in DEVICE_FILES.keys():
        path = os.path.join(DATASET_BASE_PATH, DEVICE_FILES[device])
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        for col in CATEGORICAL_FEATURES.get(device, []):
            if col in df.columns:
                df[col] = LabelEncoder().fit_transform(df[col].astype(str))
        feat_cols = [f for f in DEVICE_FEATURES[device] if f in df.columns]
        sub = df[feat_cols + ['label', 'type']].copy()
        sub['device'] = device
        dfs.append(sub)

    combined = pd.concat(dfs, ignore_index=True)
    numeric_cols = [c for c in combined.columns if c not in ['label', 'type', 'device']]
    for col in numeric_cols:
        combined[col] = pd.to_numeric(combined[col], errors='coerce')
        combined[col] = combined[col].fillna(combined[col].median())

    print(f'Combined: {len(combined)} linhas, {len(numeric_cols)} features')

    X_raw = combined[numeric_cols].values.astype(float)
    X_norm = MinMaxScaler().fit_transform(X_raw)
    y = combined['type'].astype(str).tolist()

    X_tr_n, X_te_n, y_tr, y_te = train_test_split(
        X_norm, y, test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=y)

    # ── OVERSAMPLING ──
    print('\n[Combined] Aplicando oversampling no conjunto de treino...')
    X_tr_bal, y_tr_bal = oversample_replication(X_tr_n, y_tr, random_state=RANDOM_STATE)

    # ── Pesos do combinado (mesmo critério do notebook) ──
    combined_weights = []
    for f in numeric_cols:
        peso = 1
        for dev, w in BEST_WEIGHTS.items():
            if f in w:
                peso = max(peso, w[f])
                break
        combined_weights.append(peso)
    thermo_sizes = [p * BASE_THERMO_BITS for p in combined_weights]

    dy = wp.DynamicThermometer(
        thermometerSizes=thermo_sizes,
        minimum=[0.0]*len(numeric_cols),
        maximum=[1.0]*len(numeric_cols))

    print(f'   Binarizando {len(X_tr_bal)} amostras de treino balanceadas...')
    X_tr_bin = [dy.transform(row.tolist()).list() for row in X_tr_bal]
    print(f'   Binarizando {len(X_te_n)} amostras de teste...')
    X_te_bin = [dy.transform(row.tolist()).list() for row in X_te_n]
    n_bits = len(X_tr_bin[0])
    addr = pick_address_size(n_bits, preferred=ADDRESS_SIZE)
    print(f'   N_BITS={n_bits}, addressSize={addr}')

    results = []
    for model_name in models:
        print(f'\n--- Treinando {model_name} (oversampled, Combined) ---')
        if model_name == 'WiSARD':
            m = wp.Wisard(addr, bleachingActivated=True)
        else:
            m = wp.ClusWisard(addr, CLUS_MIN_SCORE, CLUS_THRESHOLD, CLUS_LIMIT)

        t0 = time.time()
        m.train(wp.DataSet(X_tr_bin, y_tr_bal))
        train_time = time.time() - t0

        t0 = time.time()
        y_pred = list(m.classify(wp.DataSet(X_te_bin)))
        test_time = time.time() - t0

        r = evaluate(y_te, y_pred, f'{model_name}-OS-Combined', 'multiclass', train_time, test_time)
        r['device'] = 'Combined'
        r['oversampled'] = True
        results.append(r)

    return results


# ============================================================
# EXECUÇÃO
# ============================================================
print('\n' + '#'*70)
print('# EXPERIMENTO DE OVERSAMPLING — Weather + Combined')
print('# Modelos: WiSARD, ClusWiSARD')
print('# Tarefa: multi-classe (binário não precisa de balanceamento)')
print('#'*70)

oversample_results = []

# Weather
oversample_results.extend(
    run_oversampling_experiment_device('Weather', models=('WiSARD', 'ClusWiSARD'))
)

# Combined
oversample_results.extend(
    run_oversampling_experiment_combined(models=('WiSARD', 'ClusWiSARD'))
)

# ============================================================
# SUMÁRIO FINAL — fácil de copiar para a apresentação
# ============================================================
print('\n\n' + '='*70)
print('SUMÁRIO FINAL — RESULTADOS DE OVERSAMPLING')
print('='*70)
print('Copie o bloco abaixo e cole na conversa:\n')
print('---BEGIN_OVERSAMPLE_RESULTS---')
for r in oversample_results:
    print(f"{r['device']:10s} | {r['model']:25s} | "
          f"acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  "
          f"rec={r['recall']:.4f}  f1={r['f1']:.4f}  "
          f"train={r['train_time']:.2f}s  test={r['test_time']:.2f}s")
print('---END_OVERSAMPLE_RESULTS---')

# Salvar CSV também
df_os = pd.DataFrame(oversample_results)
df_os.to_csv('results_oversampling.csv', index=False)
print('\nSalvo: results_oversampling.csv')

# ============================================================
# (Opcional) Relatório por classe para ver se classes raras melhoraram
# ============================================================
from sklearn.metrics import classification_report

print('\n\n=== Detalhes por classe — para conferir recall em classes raras ===')
print('(Os classification_reports detalhados foram impressos acima em cada evaluate())')